# LSTM one-month-ahead TWSA forecast with prediction intervals

Runs `scripts/lstm_forecast.py` on Google Colab or locally. The model, settings and outputs are described in the README section *LSTM forecaster with prediction intervals*; settings are in `configs/lstm_config.yaml`.

A GPU is optional. On Colab choose **Runtime → Change runtime type → T4 GPU** for a run of a few minutes; on CPU the five-member ensemble takes roughly an hour.

In [ ]:
# Colab: mount Google Drive and work inside My Drive/DATA 255 Group Project, so outputs persist.
# If the repository is not there yet, it is cloned from GitHub (the GRACE-FO split files are tracked in Git).
import os, sys
DRIVE_FOLDER = "/content/drive/MyDrive/DATA 255 Group Project"
REPO = "Multimodal-Deep-Learning-for-Reliable-GRACE-Water-Storage-Trends"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    os.chdir(DRIVE_FOLDER)
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Lyra0710/{REPO}.git
    os.chdir(REPO)
    !git pull -q
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working in", os.getcwd())
!pip -q install netCDF4 PyYAML

In [ ]:
import torch
print("PyTorch", torch.__version__, "| CUDA GPU:", torch.cuda.is_available(), "| Apple GPU:", torch.backends.mps.is_available())

## Tests (leakage checks, metrics, conformal coverage)

In [ ]:
!python -m unittest discover -s tests -p test_lstm_forecast.py -v

## Train the ensemble and evaluate on the 2024–2025 test months

In [ ]:
!python scripts/lstm_forecast.py --config configs/lstm_config.yaml

## Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

m = "results/metrics/"
display(pd.read_csv(m + "lstm_test_metrics.csv").round(3))
display(pd.read_csv(m + "lstm_test_metrics_by_year.csv").pivot(index="model", columns="year", values="rmse_cm").round(3))
display(pd.read_csv(m + "lstm_region_metrics.csv").pivot(index="region", columns="model", values="rmse_cm").round(2))
display(pd.read_csv(m + "lstm_interval_coverage.csv").round(3))
print(json.dumps(json.load(open(m + "lstm_uncertainty_summary.json")), indent=2))

In [ ]:
for name in ["test_actual_vs_predicted", "rmse_by_test_month", "skill_maps", "interval_reliability"]:
    display(Image(f"results/figures/lstm_{name}.png"))